# Assignment 3 - Minimal PPO (TRL Experimental + LoRA)

This notebook runs a small PPO stage on top of the SFT model from Assignment 1, using the reward model from Assignment 2.

Design goals:
- small enough to run on Google Colab
- reuse the saved SFT and reward-model adapters
- keep the PPO setup close to the standard RLHF pipeline

What this notebook uses:
- policy: the SFT LoRA adapter
- reward model: the reward-model LoRA adapter, frozen
- value model: another copy of the reward-model adapter, trainable
- reference policy: implicit, by disabling the PPO adapter during KL computation

> Note: PPO in `trl==0.29.0` lives under `trl.experimental`. The API is unstable, so this notebook intentionally stays minimal.


## Step 1: Install packages

Run this first in Colab. If Colab asks for a runtime restart, restart and run the notebook again.


In [ ]:
!pip -q install -U "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
                  "trl==0.29.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "wandb" "evaluate" "tensorboard"

## Step 2: Mount Google Drive

The notebook expects the SFT and reward-model outputs from the first two assignments to already exist in Google Drive.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive is ready.')
except ImportError:
    print('This notebook is not running in Google Colab.')


## Step 3: Imports, paths, and basic options

This cell sets up a small PPO run that is realistic enough to demonstrate the RLHF step, but still light enough for Colab.


In [ ]:
import gc
import json
import os
import random
from datetime import datetime

import numpy as np
import torch
from datasets import load_dataset
from peft import AutoPeftModelForCausalLM, AutoPeftModelForSequenceClassification
from transformers import AutoTokenizer, set_seed

os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

from trl.experimental.ppo.ppo_config import PPOConfig
from trl.experimental.ppo.ppo_trainer import PPOTrainer

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# -----------------------
# Paths
# -----------------------
IN_COLAB = "COLAB_GPU" in os.environ
DRIVE_ROOT = "/content/drive/MyDrive/RLHF4LLMs"
SFT_RUN_NAME = "sft_baseline_lora"
RM_RUN_NAME = "reward_model_hh_rlhf_lora"
RUN_NAME = "ppo_minimal_lora"

if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    raise ValueError("Please run the Google Drive mount cell first.")

BASE_ROOT = DRIVE_ROOT if IN_COLAB else os.path.abspath(".")
BASE_DIR = os.path.join(BASE_ROOT, RUN_NAME)
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "final_model")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
SFT_MODEL_DIR = os.path.join(BASE_ROOT, SFT_RUN_NAME, "final_model")
REWARD_MODEL_DIR = os.path.join(BASE_ROOT, RM_RUN_NAME, "final_model")

for path in [BASE_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)


# -----------------------
# Model and data
# -----------------------
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PROMPT_DATASET = "tatsu-lab/alpaca"

# -----------------------
# Small Colab-friendly PPO run
# -----------------------
FAST_MODE = True

if FAST_MODE:
    MAX_TRAIN_PROMPTS = 128
    MAX_EVAL_PROMPTS = 16
    MAX_PROMPT_LENGTH = 192
    RESPONSE_LENGTH = 96
    TOTAL_EPISODES = 64
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 4
    NUM_MINI_BATCHES = 1
    NUM_PPO_EPOCHS = 1
    LEARNING_RATE = 1e-5
    LOGGING_STEPS = 1
    SAVE_STEPS = 8
else:
    MAX_TRAIN_PROMPTS = 512
    MAX_EVAL_PROMPTS = 32
    MAX_PROMPT_LENGTH = 256
    RESPONSE_LENGTH = 128
    TOTAL_EPISODES = 256
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    NUM_MINI_BATCHES = 1
    NUM_PPO_EPOCHS = 2
    LEARNING_RATE = 1e-5
    LOGGING_STEPS = 1
    SAVE_STEPS = 16

EVAL_PROMPTS_TO_SHOW = 5
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

print("Base folder:", BASE_DIR)
print("SFT model:", SFT_MODEL_DIR)
print("Reward model:", REWARD_MODEL_DIR)
print("TensorBoard logs:", TB_LOG_DIR)
print("FAST_MODE:", FAST_MODE)
print("MODEL_DTYPE:", MODEL_DTYPE)
print("cuda:", torch.cuda.is_available())

## Step 4: Check prerequisite models

PPO needs both the saved SFT adapter and the saved reward-model adapter.


In [ ]:
required_files = [
    os.path.join(SFT_MODEL_DIR, "adapter_config.json"),
    os.path.join(REWARD_MODEL_DIR, "adapter_config.json"),
]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        "Missing required adapter files. Run Assignment 1 and Assignment 2 first. Missing:\n" + "\n".join(missing)
    )

print("Found SFT and reward-model adapters.")

## Step 5: Load tokenizer and prompt dataset

PPO trains on prompts only. The model will generate responses online, and the reward model will score those responses.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

raw_prompts = load_dataset(PROMPT_DATASET, split="train")

def format_prompt(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    if input_text:
        prompt = (
            "Below is an instruction and an input. Write a helpful response.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "Below is an instruction. Write a helpful response.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )
    return {"prompt": prompt}

prompt_ds = raw_prompts.map(format_prompt, remove_columns=raw_prompts.column_names)
prompt_ds = prompt_ds.shuffle(seed=SEED)
train_prompt_ds = prompt_ds.select(range(min(MAX_TRAIN_PROMPTS, len(prompt_ds))))
eval_prompt_ds = train_prompt_ds.select(range(min(MAX_EVAL_PROMPTS, len(train_prompt_ds))))

def tokenize_prompt(example):
    tokenized = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
        padding=False,
    )
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
    }

ppo_train_ds = train_prompt_ds.map(tokenize_prompt, remove_columns=train_prompt_ds.column_names)
ppo_eval_ds = eval_prompt_ds.map(tokenize_prompt, remove_columns=eval_prompt_ds.column_names)
sample_prompts = eval_prompt_ds["prompt"][:EVAL_PROMPTS_TO_SHOW]

dataset_info = {
    "prompt_dataset": PROMPT_DATASET,
    "train_rows": len(ppo_train_ds),
    "eval_rows": len(ppo_eval_ds),
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "response_length": RESPONSE_LENGTH,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "dataset_info.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("Train prompts:", len(ppo_train_ds))
print("Eval prompts:", len(ppo_eval_ds))
print(sample_prompts[0][:400])

## Step 6: Load policy, reward model, and value model

The reward model stays frozen. The policy and value model are trainable. To keep memory practical, all three models are LoRA adapters on top of the same small base model family.


In [ ]:
policy_model = AutoPeftModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    is_trainable=True,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
policy_model.config.pad_token_id = tokenizer.pad_token_id
policy_model.config.use_cache = False
policy_model.gradient_checkpointing_enable()

reward_model = AutoPeftModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_DIR,
    is_trainable=False,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.config.use_cache = False
reward_model.eval()
for param in reward_model.parameters():
    param.requires_grad = False

value_model = AutoPeftModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_DIR,
    is_trainable=True,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
value_model.config.pad_token_id = tokenizer.pad_token_id
value_model.config.use_cache = False
value_model.gradient_checkpointing_enable()

print("Policy trainable params:")
policy_model.print_trainable_parameters()
print("Value model trainable params:")
value_model.print_trainable_parameters()

## Step 7: Build PPO trainer

This uses the experimental PPO trainer from TRL. Because the policy is a PEFT model, we can set `ref_model=None` and let the trainer use the base model without the adapter as the KL reference.


In [ ]:
ppo_args = PPOConfig(
    output_dir=CHECKPOINT_DIR,
    run_name=RUN_NAME,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_mini_batches=NUM_MINI_BATCHES,
    num_ppo_epochs=NUM_PPO_EPOCHS,
    total_episodes=TOTAL_EPISODES,
    response_length=RESPONSE_LENGTH,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to=["tensorboard"],
    logging_dir=TB_LOG_DIR,
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    temperature=0.9,
    kl_coef=0.05,
    num_sample_generations=0,
    disable_tqdm=False,
)

trainer = PPOTrainer(
    args=ppo_args,
    processing_class=tokenizer,
    model=policy_model,
    ref_model=None,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_train_ds,
    eval_dataset=ppo_eval_ds,
)

config_info = {
    "run_name": RUN_NAME,
    "sft_model_dir": SFT_MODEL_DIR,
    "reward_model_dir": REWARD_MODEL_DIR,
    "total_episodes": TOTAL_EPISODES,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_mini_batches": NUM_MINI_BATCHES,
    "num_ppo_epochs": NUM_PPO_EPOCHS,
    "response_length": RESPONSE_LENGTH,
    "learning_rate": LEARNING_RATE,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "ppo_config.json"), "w", encoding="utf-8") as f:
    json.dump(config_info, f, ensure_ascii=False, indent=2)

print(config_info)

## Step 8: Optional TensorBoard view

The event files are written to `TB_LOG_DIR` via `TENSORBOARD_LOGGING_DIR`.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

## Step 9: Generate a few samples before PPO

These prompts are reused after PPO so you can compare the policy qualitatively.


In [ ]:
def generate_response(model, prompt, max_new_tokens=RESPONSE_LENGTH):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

before_samples = []
for prompt in sample_prompts:
    before_samples.append({
        "prompt": prompt,
        "response": generate_response(policy_model, prompt),
    })

with open(os.path.join(RESULTS_DIR, "before_ppo_samples.json"), "w", encoding="utf-8") as f:
    json.dump(before_samples, f, ensure_ascii=False, indent=2)

before_samples[0]

## Step 10: Train with PPO

This is the actual RLHF step: sample responses from the policy, score them with the reward model, add a KL penalty to stay close to the SFT policy, and update the policy and value model.


In [ ]:
trainer.train()

ppo_log_history = trainer.state.log_history
with open(os.path.join(RESULTS_DIR, "ppo_log_history.json"), "w", encoding="utf-8") as f:
    json.dump(ppo_log_history, f, ensure_ascii=False, indent=2)

print("Logged entries:", len(ppo_log_history))
if ppo_log_history:
    print(ppo_log_history[-1])

## Step 11: Compare sample outputs after PPO

The goal here is not a rigorous benchmark, just a quick sanity check that the trained policy still responds coherently and has changed after PPO.


In [ ]:
after_samples = []
for prompt in sample_prompts:
    after_samples.append({
        "prompt": prompt,
        "response": generate_response(policy_model, prompt),
    })

comparison_rows = []
for before_item, after_item in zip(before_samples, after_samples):
    comparison_rows.append({
        "prompt": before_item["prompt"],
        "before_ppo": before_item["response"],
        "after_ppo": after_item["response"],
    })

comparison_path = os.path.join(RESULTS_DIR, "ppo_sample_comparison.json")
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, ensure_ascii=False, indent=2)

print("Saved sample comparison to:", comparison_path)
comparison_rows[0]

## Step 12: Save the PPO policy

This saves the updated policy adapter for inference and later comparison.


In [ ]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

save_info = {
    "final_model_dir": FINAL_MODEL_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "tb_log_dir": TB_LOG_DIR,
    "results_dir": RESULTS_DIR,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "save_info.json"), "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2)

print("Saved PPO policy to:", FINAL_MODEL_DIR)
print("Saved result files to:", RESULTS_DIR)